## 20. Forward-test LightGBM без RD на новых днях Bybit (1–4 марта 2026)

**Цель:** проверить, что наша модель **без RD-фичей** (LightGBM + seq_5_15_30_60, как в ноутбуках 18/19) даёт сопоставимые
метрики `avg_%_per_trade` и разумный профиль сделок **на других датах и живых свечах Bybit**, а не только на тестовом дне
из исходного датасета.

Подход:
1. Берём тот же train/val/test сплит 1–10 февраля 2026 из `data_labeled_tp_sl_1_05.parquet`.
2. Обучаем LightGBM **без RD-фичей**, как в ноутбуке 19 (`lgb_no_rd`).
3. Выбираем топ-50 символов по длине сессий на тестовом дне (2026-02-10) — те символы, по которым модель уже показала себя.
4. Через Bybit API (V5, category=`linear`, interval=`1`) загружаем 1m-свечи по этим символам на новые дни:
   - 2026-03-01,
   - 2026-03-02,
   - 2026-03-03,
   - 2026-03-04.
5. Для каждой даты строим сессии и фичи **без RD** (только OHLCV+time), запускаем модель и считаем бэктест `signal_flip` 
   с порогом 0.75 и комиссией 0.1%.
6. Сравниваем `avg_%_per_trade` и профиль сделок по дням, чтобы убедиться, что поведение модели устойчиво и не является
   артефактом «удачно подобранного» тестового дня.

Важно:
- RD-фичей на живых данных нет, поэтому считаем **ровно тот же набор BASE_NO_RD и rolling-фичей**, что в ноутбуках 18/19.
- Для загрузки свечей используем внутренний helper из `src/features/warmup_loader.py` (`_fetch_klines_from_bybit`),
  тем самым также проверяя его работоспособность в реальном сценарии.

In [9]:
import sys, os, numpy as np, pandas as pd, warnings, datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == '05_experiments' else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.features.warmup_loader import _fetch_klines_from_bybit, _to_bybit_symbol

COMMISSION_RT = 0.001
THRESHOLD = 0.75  # band 25-75
TARGET_COL = 'target'

print('Root:', _root)

Root: c:\project\trading_bot_2Engine


### 1. Загрузка размеченных данных и базовых фичей (как в 18/19)

In [10]:
labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

valid = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'

train_dates = set(dates[:8])
val_dates   = set([dates[8]])
test_dates  = set([dates[9]])

print('Dates: train=', f'{min(train_dates)}..{max(train_dates)}', '| val=', dates[8], '| test=', dates[9])
print('Rows total:', len(valid))
print('BASE_FEATURES (22):', BASE_FEATURES)

Dates: train= 2026-02-01..2026-02-08 | val= 2026-02-09 | test= 2026-02-10
Rows total: 186063
BASE_FEATURES (22): ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'rd_regime', 'rd_regime_transition']


### 2. RD-фичи и набор BASE_NO_RD (как в 18/19)

- RD-фичи: все, что начинается с `rd_` + `abs_rd`.
- BASE_NO_RD: только OHLCV/time-признаки, доступные от биржи.

In [11]:
def is_rd_feature(c: str) -> bool:
    if c.startswith('rd_'):
        return True
    if c == 'abs_rd':
        return True
    return False

BASE_NO_RD = [c for c in BASE_FEATURES if not is_rd_feature(c)]
print('BASE_NO_RD (без RD):', len(BASE_NO_RD), BASE_NO_RD)

# Для rolling без RD используем те же фичи, что в 18-м ноутбуке
KEY_FEATS_NO_RD = [c for c in BASE_NO_RD if c in ['ret_1', 'ret_5', 'rsi_14', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position']]
SEQ_WINDOWS = [5, 15, 30, 60]
print('KEY_FEATS_NO_RD:', KEY_FEATS_NO_RD)

BASE_NO_RD (без RD): 13 ['ret_1', 'ret_5', 'rsi_14', 'macd_signal', 'macd_hist', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
KEY_FEATS_NO_RD: ['ret_1', 'ret_5', 'rsi_14', 'volatility_14', 'volume_rel_20', 'body_ratio', 'close_position']


### 3. Обучаем LightGBM без RD (на февральских данных)

Полностью воспроизводим `lgb_no_rd` из ноутбука 19: тот же split 1–8/9/10 февраля, те же BASE_NO_RD и seq_5_15_30_60 без RD.

In [12]:
# Строим rolling-фичи на valid (как в 18/19)
grp = valid.groupby('session_key', group_keys=False)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS_NO_RD:
        valid[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
        valid[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

SEQ_NO_RD = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS_NO_RD for s in ('mean', 'std')]
FEATURES_BASE_NO_RD = BASE_NO_RD[:]
FEATURES_SEQ_NO_RD = FEATURES_BASE_NO_RD + SEQ_NO_RD

train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df   = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df  = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

print('FEATURES_BASE_NO_RD:', len(FEATURES_BASE_NO_RD))
print('FEATURES_SEQ_NO_RD:', len(FEATURES_SEQ_NO_RD))
print('Rows (train/val/test):', len(train_df), len(val_df), len(test_df))

FEATURES_BASE_NO_RD: 13
FEATURES_SEQ_NO_RD: 69
Rows (train/val/test): 146693 24014 15356


In [13]:
scaler_no_rd = StandardScaler()
X_tr_no_rd = scaler_no_rd.fit_transform(train_df[FEATURES_SEQ_NO_RD].fillna(0))
X_va_no_rd = scaler_no_rd.transform(val_df[FEATURES_SEQ_NO_RD].fillna(0))
X_te_no_rd = scaler_no_rd.transform(test_df[FEATURES_SEQ_NO_RD].fillna(0))

y_tr = train_df['y'].values
y_va = val_df['y'].values
y_te = test_df['y'].values

lgb_no_rd = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
lgb_no_rd.fit(X_tr_no_rd, y_tr)

p_va = lgb_no_rd.predict_proba(X_va_no_rd)[:, 1]
p_te = lgb_no_rd.predict_proba(X_te_no_rd)[:, 1]
print('AUC no-RD: val=', roc_auc_score(y_va, p_va), 'test=', roc_auc_score(y_te, p_te))

AUC no-RD: val= 0.7296805752681584 test= 0.7126151138873442


### 4. Топ-50 символов по длине сессий на тестовом дне (2026-02-10)

Выбираем символы, по которым в исходном датасете у нас больше всего минуток на дне 2026-02-10 — их и будем forward-тестить на новых датах.

In [14]:
test_date = list(test_dates)[0]
df_test_day = valid[valid['date'] == test_date].copy()
sym_counts = df_test_day.groupby('symbol').size().sort_values(ascending=False)
top50_symbols = sym_counts.head(50).index.tolist()
print('Top 50 symbols by rows on test day:', len(top50_symbols))
top50_symbols[:10]

Top 50 symbols by rows on test day: 50


['WHITEWHALE',
 'POWER',
 'PIPPIN',
 'GPS',
 'ROAM',
 'ZKP',
 'BEAT',
 'STABLE',
 'XNY',
 'RIVER']

### 5. Загрузка 1m-свечей с Bybit для 1–4 марта 2026

- Используем `_fetch_klines_from_bybit` из `warmup_loader`.
- Для каждого символа и даты запрашиваем последний `N` баров до конца дня (limit ≥ 1440), затем фильтруем по дате.
- Работает на `category='linear'`, `interval='1'`.

Для реального запуска нужны корректные env-переменные BYBIT_API_KEY/BYBIT_API_SECRET (для больших лимитов и защиты от rate-limit).

In [15]:
def load_bybit_day(symbol: str, date: dt.date, limit: int = 1500) -> pd.DataFrame:
    """Загружает 1m Kline по символу за указанный день (UTC) через Bybit V5.
    Берёт последние `limit` свечей до конца дня и фильтрует по дате.
    """
    symbol_bybit = _to_bybit_symbol(symbol)
    # конец дня в UTC
    end_dt = dt.datetime(date.year, date.month, date.day, 23, 59, 59, tzinfo=dt.timezone.utc)
    end_ms = int(end_dt.timestamp() * 1000)
    df = _fetch_klines_from_bybit(symbol_bybit, end_ts_ms=end_ms, limit=limit)
    if df.empty:
        return df
    df['datetime'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df['date'] = df['datetime'].dt.date
    df = df[df['date'] == date].copy()
    df['symbol'] = symbol
    return df.sort_values('datetime').reset_index(drop=True)

forward_dates = [dt.date(2026, 3, d) for d in [1, 2, 3, 4]]
forward_dates

[datetime.date(2026, 3, 1),
 datetime.date(2026, 3, 2),
 datetime.date(2026, 3, 3),
 datetime.date(2026, 3, 4)]

### 6. Построение фичей без RD и бэктест по каждому дню

Для каждой даты:

1. Загружаем свечи по top-50 символам.
2. Формируем сессии: одна сессия = (symbol, day), т.к. данные плотные.
3. Строим BASE_NO_RD и rolling-фичи 5/15/30/60 по KEY_FEATS_NO_RD.
4. Прогоняем через обученный `lgb_no_rd` и считаем бэктест `signal_flip` с порогом 0.75.
5. Сохраняем `net_%`, `trades`, `avg_%_per_trade` для каждой даты (агрегировано по всем символам).

In [17]:
def backtest_prod_hold(proba, ret, session_ids, threshold=THRESHOLD, commission_rt=COMMISSION_RT):
    """signal_flip: BUY >= thr, SELL <= 1-thr, HOLD держим позицию."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt_ = pd.to_datetime(df['datetime'], utc=True)
    hour = dt_.dt.hour + dt_.dt.minute / 60
    dow = dt_.dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)
    return df

def add_ohlc_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rng = df['high'] - df['low']
    rng = rng.replace(0, np.nan)
    body = (df['close_price'] - df['open']).abs()
    df['body_ratio'] = (body / rng).fillna(0.5).clip(0, 1)
    df['close_position'] = ((df['close_price'] - df['low']) / rng).fillna(0.5).clip(0, 1)
    return df

def add_price_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby('session_key', group_keys=False)
    close = grp['close_price']
    df['ret_1'] = close.pct_change(1)
    df['ret_5'] = close.pct_change(5)
    # RSI(14)
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.rolling(14, min_periods=1).mean()
    avg_loss = loss.rolling(14, min_periods=1).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan).fillna(1e-9))
    df['rsi_14'] = 100 - (100 / (1 + rs))
    # MACD(12,26,9) — как в feature_pipeline
    ema12 = close.transform(lambda x: x.ewm(span=12, adjust=False).mean())
    ema26 = close.transform(lambda x: x.ewm(span=26, adjust=False).mean())
    df['macd_line'] = ema12 - ema26
    df['macd_signal'] = df.groupby('session_key', group_keys=False)['macd_line'].transform(
        lambda x: x.ewm(span=9, adjust=False).mean()
    )
    df['macd_hist'] = df['macd_line'] - df['macd_signal']
    # volatility_14
    df['volatility_14'] = close.pct_change().rolling(14, min_periods=1).std()
    # volume_rel_20
    vol_ma = grp['volume'].transform(lambda x: x.rolling(20, min_periods=1).mean())
    df['volume_rel_20'] = df['volume'] / (vol_ma + 1e-9)
    return df

def build_features_no_rd(df: pd.DataFrame) -> pd.DataFrame:
    df = add_time_features(df)
    df = add_ohlc_features(df)
    # session_key уже есть
    df = add_price_volume_features(df)
    # rolling
    grp = df.groupby('session_key', group_keys=False)
    for w in SEQ_WINDOWS:
        for c in KEY_FEATS_NO_RD:
            df[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f'{c}_roll{w}_std']  = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    return df

def forward_test_day(target_date: dt.date) -> dict:
    frames = []
    for sym in top50_symbols:
        df_sym = load_bybit_day(sym, target_date)
        if df_sym.empty:
            continue
        df_sym['session_key'] = df_sym['symbol'].astype(str) + '_' + df_sym['date'].astype(str)
        frames.append(df_sym)
    if not frames:
        return {'date': target_date, 'rows': 0, 'net_%': np.nan, 'trades': 0, 'avg_%_per_trade': np.nan}
    df_day = pd.concat(frames, ignore_index=True).sort_values(['session_key', 'datetime']).reset_index(drop=True)

    # рассчитываем ret_next
    df_day['ret_next'] = df_day.groupby('session_key')['close_price'].pct_change().shift(-1)
    df_day = df_day.dropna(subset=['ret_next']).reset_index(drop=True)

    # строим фичи без RD
    df_feat = build_features_no_rd(df_day)
    # отберём только нужные фичи
    cols = [c for c in FEATURES_SEQ_NO_RD if c in df_feat.columns]
    X = scaler_no_rd.transform(df_feat[cols].fillna(0))
    proba = lgb_no_rd.predict_proba(X)[:, 1]
    bt = backtest_prod_hold(proba, df_feat['ret_next'].values, df_feat['session_key'].values, threshold=THRESHOLD)
    return {
        'date': target_date,
        'rows': len(df_feat),
        'net_%': bt['net_%'],
        'trades': bt['trades'],
        'avg_%_per_trade': bt['avg_%_per_trade'],
    }

forward_results = [forward_test_day(d) for d in forward_dates]
forward_df = pd.DataFrame(forward_results)
forward_df

,date,rows,net_%,trades,avg_%_per_trade
0,2026-03-01,48951,22.748733,426,0.053401
1,2026-03-02,48951,43.528458,544,0.080016
2,2026-03-03,48951,-151.646258,533,-0.284515
3,2026-03-04,48951,-73.815777,531,-0.139013


### 7. Как интерпретировать результаты forward-теста

Таблица `forward_df` показывает для каждого из дней 1–4 марта 2026 года:

- `rows` — сколько баров (по всем top-50 символам) попало в бэктест;
- `net_%` — суммарный PnL в процентах (по ret_next, с учётом комиссии 0.1% RT);
- `trades` — количество совершённых сделок (смен позиций);
- `avg_%_per_trade` — средняя прибыль на сделку.

Если на большинстве из этих дней `avg_%_per_trade` остаётся **в том же коридоре**, что и на исходном тестовом дне
(порядка 1–1.5% на сделку для модели без RD), и при этом нет катастрофического падения AUC (см. ноутбук 18/19),
это будет сильным аргументом в пользу того, что наша архитектура модели и выбор порога/логики сделок дают устойчивую
альфу, а не разовый оверфит под конкретный день.

При появлении реальных результатов можно дополнить ноутбук финальными выводами и зафиксировать их в документации.

### 8. Выводы по forward-тесту (1–4 марта, модель без RD)

По таблице `forward_df`:

- **Объём данных:**
  - `rows` одинаковы для всех дней (48 951 строк), что соответствует ~50 символам × ~1000 баров на день → данные по объёму стабильны, крупных провалов/обрезаний нет.
  - Кол-во сделок порядка 500+ в день → модель активно торгует, а не даёт единичные сигналы.

- **Результаты по дням (net_% и avg_%_per_trade):
  - 1 марта: `net_% ≈ +22.7%`, `trades=426`, `avg/trade ≈ +0.05%`.
  - 2 марта: `net_% ≈ +43.5%`, `trades=544`, `avg/trade ≈ +0.08%`.
  - 3 марта: `net_% ≈ -151.6%`, `trades=533`, `avg/trade ≈ -0.28%`.
  - 4 марта: `net_% ≈ -73.8%`,  `trades=531`, `avg/trade ≈ -0.14%`.

- **Сравнение с февральским тестовым днём (10-е число):**
  - На 10.02 модель без RD показывала `avg_%_per_trade` порядка **1.3–1.5%** на сделку.
  - На новых маршевских днях типичные значения в коридоре **от +0.05% до -0.28%** на сделку, причём половина дней уходит в уверенный минус по средней сделке.

- **Интерпретация:**
  - Forward-тест показывает, что LightGBM **без RD-фичей** малоустойчив к смене рыночного режима: на новых датах средняя прибыль на сделку сильно «сдувается» и легко становится отрицательной.
  - Это означает, что тот высокий avg/trade, который мы видели на февральском тестовом дне без RD, в значительной степени эффект конкретного рыночного окна, а не устойчивая закономерность.
  - Для реального продакшена модель без RD в текущем виде использовать рискованно: нужна либо дополнительная фильтрация по режиму рынка, либо опора на RD-блок заказчика, который в экспериментах 18–19 как раз даёт более стабильное качество и PnL.